# Phase 1 — Power Laws & Curve Fitting
### Scaling Laws for Neural Language Models (Kaplan et al., 2020)

Before diving into the paper's results, we need to build intuition for the mathematical tool the entire paper rests on: **power laws**.

The core claim of the paper is simple:

> *Loss decreases as a power law of model size, dataset size, and compute — each independently.*

This notebook covers:
1. What a power law is and why it appears on a straight line in log-log space
2. How to fit a power law using only linear regression (no scipy needed)
3. Applying this to synthetic language model loss data
4. Reproducing the visual style of Figure 1 from the paper

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
print('Libraries ready.')

## 1. What is a Power Law?

A power law has the form:

$$L = a \cdot x^{-\alpha}$$

- `L` is the loss (what we're trying to minimise)
- `x` is some resource (model size, data, compute)
- `α` is the **scaling exponent** — the bigger it is, the faster loss drops as you add resources
- `a` is a constant that shifts the curve up/down

The paper's central finding is that `α` is **consistent and predictable** across orders of magnitude.

In [ ]:
# --- Visualise power laws with different exponents ---

x = np.logspace(0, 6, 300)  # x from 1 to 1,000,000

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

alphas = [0.05, 0.076, 0.10, 0.20]
colours = ['#e74c3c', '#e67e22', '#2ecc71', '#3498db']
labels  = [f'α = {a}' for a in alphas]

# --- Left: linear axes ---
ax = axes[0]
for alpha, colour, label in zip(alphas, colours, labels):
    loss = 3.0 * x**(-alpha)
    ax.plot(x, loss, color=colour, label=label, linewidth=2)
ax.set_xlabel('Resource (e.g. parameters)')
ax.set_ylabel('Loss L')
ax.set_title('Power Laws — Linear Axes')
ax.set_xlim([0, 1e6])
ax.set_ylim([0, 3.5])
ax.legend()
ax.grid(True, alpha=0.3)

# --- Right: log-log axes ---
ax = axes[1]
for alpha, colour, label in zip(alphas, colours, labels):
    loss = 3.0 * x**(-alpha)
    ax.loglog(x, loss, color=colour, label=label, linewidth=2)
ax.set_xlabel('Resource (log scale)')
ax.set_ylabel('Loss L (log scale)')
ax.set_title('Same Power Laws — Log-Log Axes → Straight Lines!')
ax.legend()
ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.suptitle('The Magic of Log-Log Space', y=1.02, fontsize=13, fontweight='bold')
plt.show()

print("""
Key insight: On log-log axes, a power law L = a * x^(-α) becomes a straight line:
  log(L) = log(a) - α * log(x)
  
The SLOPE of that line is -α (the scaling exponent).
The INTERCEPT is log(a).

This means we can FIT power laws using simple linear regression in log-log space.
""")

## 2. Fitting a Power Law Without scipy

Since a power law is a straight line in log-log space, we can fit it with ordinary least squares:

$$\log L = \underbrace{\log a}_{\text{intercept}} - \alpha \cdot \underbrace{\log x}_{\text{input}}$$

This is `np.polyfit(log(x), log(L), 1)` — a degree-1 polynomial fit on the logs.

In [ ]:
def fit_power_law(x_data, y_data):
    """
    Fit y = a * x^(-alpha) using log-log linear regression.
    Returns (a, alpha) and the fitted y values.
    """
    log_x = np.log(x_data)
    log_y = np.log(y_data)
    
    # np.polyfit returns [slope, intercept] for degree=1
    slope, intercept = np.polyfit(log_x, log_y, 1)
    
    a     =  np.exp(intercept)  # convert log(a) back to a
    alpha = -slope              # slope = -alpha
    
    return a, alpha


def power_law(x, a, alpha):
    return a * x**(-alpha)


# --- Test: generate noisy power-law data and recover the parameters ---
TRUE_A     = 3.5
TRUE_ALPHA = 0.076  # close to the paper's N-scaling exponent

rng = np.random.default_rng(42)
x_pts = np.logspace(7, 11, 20)                           # 10M to 100B parameters
noise = rng.lognormal(mean=0, sigma=0.02, size=len(x_pts))
y_pts = power_law(x_pts, TRUE_A, TRUE_ALPHA) * noise     # add multiplicative noise

# Fit
fitted_a, fitted_alpha = fit_power_law(x_pts, y_pts)
x_smooth = np.logspace(7, 11, 300)
y_fit    = power_law(x_smooth, fitted_a, fitted_alpha)

print(f"True values  : a = {TRUE_A:.4f},  α = {TRUE_ALPHA:.4f}")
print(f"Fitted values: a = {fitted_a:.4f},  α = {fitted_alpha:.4f}")
print(f"Recovery error: α error = {abs(fitted_alpha - TRUE_ALPHA):.6f}")

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x_pts, y_pts, color='#e74c3c', s=50, zorder=5, label='Noisy data points')
ax.loglog(x_smooth, y_fit, color='#2c3e50', linewidth=2,
          label=f'Fitted: L = {fitted_a:.2f} · N^(−{fitted_alpha:.4f})')
ax.set_xlabel('Parameters N')
ax.set_ylabel('Loss L')
ax.set_title('Power Law Fitting via Log-Log Linear Regression')
ax.legend(fontsize=10)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## 3. The Paper's Key Exponents

Table 1 from Kaplan et al. reports these exponents for large language models on text:

| Scaling axis | Law | Exponent α |
|---|---|---|
| Parameters N | L(N) = (N_c / N)^α | **0.076** |
| Dataset tokens D | L(D) = (D_c / D)^β | **0.095** |
| Compute C | L(C) = (C_c / C)^γ | **0.050** |

These exponents are **small** — loss drops slowly. Doubling parameters gives only a ~5% loss reduction. That's why you need to scale by **orders of magnitude** to see real gains.

In [ ]:
# --- Reproduce the paper's exponents and show the practical implication ---

# Approximate constants from the paper (normalised so loss = 1.0 at reference point)
# These are illustrative fits matching the paper's reported exponents.
paper_params = {
    'Parameters N': {'alpha': 0.076, 'x_c': 8.8e13,  'label': 'N (parameters)'},
    'Dataset D':    {'alpha': 0.095, 'x_c': 5.4e13,  'label': 'D (tokens)'},
    'Compute C':    {'alpha': 0.050, 'x_c': 3.1e8,   'label': 'C (PF-days)'},
}

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, (title, p) in zip(axes, paper_params.items()):
    alpha = p['alpha']
    x_c   = p['x_c']
    
    # L = (x_c / x)^alpha   →  loss = 1 when x = x_c
    x = np.logspace(np.log10(x_c) - 3, np.log10(x_c) + 3, 300)
    L = (x_c / x) ** alpha
    
    ax.loglog(x / x_c, L, color='#2980b9', linewidth=2.5)
    ax.axvline(x=1, color='#e74c3c', linestyle='--', alpha=0.7, label='Reference point')
    ax.axhline(y=1, color='#e74c3c', linestyle='--', alpha=0.7)
    
    # Annotate slope
    ax.text(0.05, 0.15, f'slope = −{alpha}', transform=ax.transAxes,
            fontsize=11, color='#2c3e50',
            bbox=dict(boxstyle='round', facecolor='#ecf0f1', alpha=0.8))
    
    ax.set_xlabel(f'{p["label"]} / reference', fontsize=10)
    ax.set_ylabel('Relative Loss L')
    ax.set_title(f'L({title})')
    ax.grid(True, which='both', alpha=0.3)
    ax.legend(fontsize=9)

plt.suptitle("Paper's Three Scaling Laws — Each is a Straight Line in Log-Log Space",
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 4. Intuition: How Much Do You Gain by Scaling?

With a small exponent like 0.076, you need to scale by **a lot** to see meaningful improvements. Let's quantify this.

In [ ]:
def loss_reduction(scale_factor, alpha):
    """By what % does loss drop when you multiply resources by scale_factor?"""
    return (1 - scale_factor**(-alpha)) * 100

print("How much does loss DROP when you increase parameters by a factor of X?")
print(f"(Using α = 0.076 from the paper)")
print("-" * 50)

factors = [2, 10, 100, 1_000, 10_000, 1_000_000]
alpha_N = 0.076

for f in factors:
    reduction = loss_reduction(f, alpha_N)
    print(f"  {f:>10,}x more parameters  →  {reduction:5.1f}% loss reduction")

print()
print("Key insight: you need 10,000x more parameters just to cut loss by ~47%.")
print("Scaling is powerful but requires enormous resource multipliers.")

In [ ]:
# --- Visual: 'How much does doubling help?' across the three axes ---

multipliers = np.logspace(0, 6, 300)  # 1x to 1,000,000x

alphas_plot = {'Parameters (α=0.076)': 0.076,
               'Dataset (α=0.095)':    0.095,
               'Compute (α=0.050)':    0.050}
colours_plot = ['#3498db', '#2ecc71', '#e67e22']

fig, ax = plt.subplots(figsize=(9, 5))

for (label, alpha), colour in zip(alphas_plot.items(), colours_plot):
    reductions = [loss_reduction(m, alpha) for m in multipliers]
    ax.semilogx(multipliers, reductions, label=label, color=colour, linewidth=2)

ax.axvline(x=10,   color='grey', linestyle=':', alpha=0.6, label='10x')
ax.axvline(x=1000, color='grey', linestyle='--', alpha=0.6, label='1,000x')
ax.set_xlabel('Resource multiplier (log scale)')
ax.set_ylabel('Loss reduction (%)')
ax.set_title('Loss Reduction vs Resource Multiplier')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 100])
plt.tight_layout()
plt.show()

## 5. R² — How Well Do Power Laws Actually Fit?

The paper's big empirical claim is that power laws fit over **many orders of magnitude** with very high R². Let's verify what that looks like.

In [ ]:
def r_squared_loglog(x_data, y_data, a, alpha):
    """R² computed in log-log space (where the fit is linear)."""
    log_y_true = np.log(y_data)
    log_y_pred = np.log(power_law(x_data, a, alpha))
    ss_res = np.sum((log_y_true - log_y_pred)**2)
    ss_tot = np.sum((log_y_true - np.mean(log_y_true))**2)
    return 1 - ss_res / ss_tot


# Simulate increasingly large ranges and check how well the fit holds
print("R² of power law fit over different ranges of N:")
print("-" * 55)

ranges = [
    ('3 orders of magnitude',  7, 10),
    ('4 orders of magnitude',  7, 11),
    ('5 orders of magnitude',  7, 12),
    ('6 orders of magnitude',  7, 13),
    ('7 orders of magnitude',  7, 14),  # roughly what the paper spans
]

rng = np.random.default_rng(42)
for label, lo, hi in ranges:
    x_test = np.logspace(lo, hi, 40)
    noise  = rng.lognormal(mean=0, sigma=0.03, size=len(x_test))
    y_test = power_law(x_test, TRUE_A, TRUE_ALPHA) * noise
    a_fit, alpha_fit = fit_power_law(x_test, y_test)
    r2 = r_squared_loglog(x_test, y_test, a_fit, alpha_fit)
    print(f"  {label:<28}:  R² = {r2:.6f}")

print()
print("The paper reports R² > 0.999 across 7+ orders of magnitude.")
print("This is what makes the result remarkable — power laws don't 'break' at scale.")

## Key Takeaways — Phase 1

| Concept | What you learned |
|---|---|
| Power law form | `L = a · x^(−α)` — loss as a function of any single resource |
| Log-log linearity | Power laws become straight lines on log-log axes; slope = −α |
| Fitting | `np.polyfit(log(x), log(y), 1)` recovers α and a from noisy data |
| Kaplan exponents | N: 0.076, D: 0.095, C: 0.050 — all small, so big multipliers needed |
| Predictability | R² > 0.999 over 7 orders of magnitude — the law holds at every scale |

---

**Phase 2 →** We'll use these tools to explore all three laws together: L(N), L(D), and L(C), and see how they combine.